In [2]:
import httpx
print(f"Version installée : {httpx.__version__}")

# Si cela affiche 0.27.0, alors la commande suivante marchera :
from datasets import Dataset
print("✅ VICTOIRE ! L'erreur a disparu.")

Version installée : 0.27.0
✅ VICTOIRE ! L'erreur a disparu.


In [2]:
import pandas as pd
import ast
import re
import os
from datasets import Dataset
from transformers import AutoTokenizer

# ==========================================
# 1. CONFIGURATION
# ==========================================
CSV_PATH = "../data/raw/Dataset_Python_Question_Answer.csv"
OUTPUT_DIR = "../data/processed"
MODEL_CHECKPOINT = "bert-base-uncased" # Modèle standard pour l'anglais

print(f"🚀 Démarrage du pipeline de Nettoyage...")

# ==========================================
# 2. FONCTION DE NETTOYAGE (Deep Learning Cleaning)
# ==========================================
def clean_text_deep_learning(text):
    """
    Nettoie le texte brut pour le rendre digeste par le modèle BERT.
    """
    if not isinstance(text, str): return ""

    # 1. Gestion des listes Python stockées en texte "['...']"
    # Votre dataset Kaggle a ce format bizarre, on le corrige.
    if text.strip().startswith("['") and text.strip().endswith("']"):
        try:
            parsed = ast.literal_eval(text)
            text = " ".join(parsed) # On recolle les morceaux
        except:
            pass 

    # 2. Suppression du bruit Markdown (Gras, Code, Liens)
    text = re.sub(r'\*\*|__', '', text)       # Enlève le gras (**mot**)
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL) # Enlève les blocs de code
    text = re.sub(r'`', '', text)             # Enlève les petits codes `var`
    text = re.sub(r'\[.*?\]', '', text)       # Enlève les références [1]
    
    # 3. Suppression du bruit "Conversationnel"
    # On enlève les "Sure!", "Here is..." qui n'apportent rien à l'IA
    text = re.sub(r"^Sure.*?[:\.]", "", text, flags=re.IGNORECASE)
    
    # 4. Standardisation
    text = text.lower() # Minuscule
    text = re.sub(r'\s+', ' ', text).strip() # Espaces en trop
    
    return text

# ==========================================
# 3. CHARGEMENT ET NETTOYAGE DU CSV
# ==========================================
print(f"📂 Chargement du fichier : {CSV_PATH}")

try:
    df = pd.read_csv(CSV_PATH)
    
    # Renommage des colonnes pour être standard
    df.columns = [c.lower() for c in df.columns] 
    
    # Vérification des colonnes
    if 'question' not in df.columns or 'answer' not in df.columns:
        raise ValueError(f"Colonnes attendues 'Question' et 'Answer' non trouvées. Colonnes actuelles : {df.columns}")
        
    print("🧹 Nettoyage du texte en cours (Regex + Parsing)...")
    # On applique le nettoyage sur les questions et les réponses
    df['clean_question'] = df['question'].apply(clean_text_deep_learning)
    df['clean_context'] = df['answer'].apply(clean_text_deep_learning)
    
    # On supprime les lignes vides ou trop courtes (déchets)
    initial_len = len(df)
    df = df[df['clean_context'].str.len() > 15]
    print(f"✅ Nettoyage terminé. {len(df)} lignes conservées sur {initial_len}.")
    
    # Sauvegarde du CSV PROPRE pour vérification humaine
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
    df.to_csv(f"{OUTPUT_DIR}/dataset_cleaned_readable.csv", index=False)
    print(f"📄 Fichier lisible sauvegardé : {OUTPUT_DIR}/dataset_cleaned_readable.csv")

except Exception as e:
    print(f"❌ ERREUR CRITIQUE : {e}")
    raise

# ==========================================
# 4. TOKENIZATION (Préparation Mathématique)
# ==========================================
print(f"🧠 Chargement du Tokenizer : {MODEL_CHECKPOINT}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# Conversion en objet Dataset HuggingFace
raw_dataset = Dataset.from_pandas(df)
# Split Train (80%) / Test (20%)
raw_dataset = raw_dataset.train_test_split(test_size=0.2)

def tokenize_function(examples):
    """Transforme le texte en nombres pour le GPU"""
    # On tokenise Question + Contexte
    tokenized_inputs = tokenizer(
        examples["clean_question"],
        examples["clean_context"],
        max_length=384,
        truncation="only_second", # On coupe la réponse si trop longue, pas la question
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Calcul des positions (Start/End) pour le modèle QA
    # Ici, la réponse = tout le contexte.
    start_positions = []
    end_positions = []

    for i, _ in enumerate(tokenized_inputs["input_ids"]):
        sequence_ids = tokenized_inputs.sequence_ids(i)
        
        # Trouver le début du contexte (après la question)
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        
        # Trouver la fin du contexte
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        start_positions.append(context_start)
        end_positions.append(context_end)

    tokenized_inputs["start_positions"] = start_positions
    tokenized_inputs["end_positions"] = end_positions
    return tokenized_inputs

print("⚙️ Tokenization en cours...")
tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)

# ==========================================
# 5. SAUVEGARDE FINALE (Pour l'étape suivante)
# ==========================================
save_path = f"{OUTPUT_DIR}/tokenized_data"
tokenized_datasets.save_to_disk(save_path)

print("-" * 30)
print(f"✅ SUCCÈS TOTAL ! Vos données sont prêtes.")
print(f"📂 Dataset Deep Learning sauvegardé dans : {save_path}")
print("👉 Étape suivante : Lancer l'entraînement (Trainer) sur ce dossier.")

🚀 Démarrage du pipeline de Nettoyage...
📂 Chargement du fichier : ../data/raw/Dataset_Python_Question_Answer.csv
🧹 Nettoyage du texte en cours (Regex + Parsing)...
✅ Nettoyage terminé. 171 lignes conservées sur 419.
📄 Fichier lisible sauvegardé : ../data/processed/dataset_cleaned_readable.csv
🧠 Chargement du Tokenizer : bert-base-uncased
⚙️ Tokenization en cours...


Saving the dataset (1/1 shards): 100%|████████████████████████████████████████████████████████████████████████| 35/35 [00:00<00:00, 1418.83 examples/s]

------------------------------
✅ SUCCÈS TOTAL ! Vos données sont prêtes.
📂 Dataset Deep Learning sauvegardé dans : ../data/processed/tokenized_data
👉 Étape suivante : Lancer l'entraînement (Trainer) sur ce dossier.
